## Local modules

The script imports these files, so they are written next to the notebook first.

In [ ]:
%%writefile utils.py
"""Shared helpers for the road damage detection scripts.

Not meant to be run directly. The other scripts in this folder import from it.
"""
from __future__ import annotations

import json
import random
from pathlib import Path

import numpy as np
import yaml

PROJECT_ROOT = Path(__file__).resolve().parents[1]
CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
DEFAULT_THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.05), 2)


# --------------------------------------------------------------------------- #
# General helpers
# --------------------------------------------------------------------------- #
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def save_json(obj, path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=float)


def resolve_weights(weights) -> str:
    """Use the path as given, or relative to the project root. Otherwise return it
    unchanged (so names like 'yolov8s.pt' can be auto-downloaded by Ultralytics)."""
    p = Path(weights)
    if p.exists():
        return str(p)
    if (PROJECT_ROOT / p).exists():
        return str(PROJECT_ROOT / p)
    return str(weights)


def _is_rtdetr(weights: str) -> bool:
    if "rtdetr" in weights.lower() or "rt-detr" in weights.lower():
        return True
    p = Path(weights)
    if p.suffix == ".pt" and p.exists():
        try:
            import torch

            ckpt = torch.load(str(p), map_location="cpu", weights_only=False)
            m = ckpt.get("ema") or ckpt.get("model")
            return type(m).__name__ == "RTDETRDetectionModel"
        except Exception:
            return False
    return False


def load_model(weights, task: str | None = "detect"):
    """Load a YOLO or RT-DETR model. RT-DETR needs its own class because it uses a
    different predictor, so it is detected from the file name or the checkpoint."""
    from ultralytics import RTDETR, YOLO

    weights = resolve_weights(weights)
    if _is_rtdetr(weights):
        return RTDETR(weights)
    return YOLO(weights, task=task) if task else YOLO(weights)


# --------------------------------------------------------------------------- #
# Dataset yaml helpers
# --------------------------------------------------------------------------- #
def resolve_data_yaml(data) -> Path:
    """Write a copy of the dataset yaml with an absolute `path:`.

    Ultralytics resolves relative dataset paths against its own datasets directory,
    not the current folder, so we always hand it an absolute path.
    """
    data_path = Path(data)
    if not data_path.exists():
        data_path = PROJECT_ROOT / data
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset yaml not found: {data}")
    with open(data_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    root = Path(cfg.get("path", data_path.parent))
    if not root.is_absolute():
        root = (PROJECT_ROOT / root).resolve()
    cfg["path"] = root.as_posix()
    out_dir = PROJECT_ROOT / "runs" / "_resolved"
    out_dir.mkdir(parents=True, exist_ok=True)
    out = out_dir / data_path.name
    with open(out, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out


def load_names(data_yaml) -> dict[int, str]:
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    names = cfg["names"]
    if isinstance(names, list):
        return {i: n for i, n in enumerate(names)}
    return {int(k): v for k, v in names.items()}


def get_split_dirs(data_yaml, split: str) -> tuple[Path, Path]:
    """Return (images_dir, labels_dir) for a split of a resolved dataset yaml."""
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if split not in cfg:
        raise KeyError(f"Split '{split}' is not defined in {data_yaml}")
    rel = cfg[split]
    if isinstance(rel, list):
        rel = rel[0]
    img_dir = Path(cfg["path"]) / rel
    parts = list(img_dir.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"
            break
    else:
        raise ValueError(f"Cannot derive the labels folder from {img_dir}")
    return img_dir, Path(*parts)


def list_images(folder) -> list[Path]:
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMG_EXTS)


def read_yolo_labels(path) -> tuple[np.ndarray, np.ndarray]:
    """Read a YOLO label file. Returns (class_ids[n], boxes_xyxy_normalised[n, 4])."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return np.zeros(0, dtype=int), np.zeros((0, 4))
    arr = np.loadtxt(path, ndmin=2)
    cls = arr[:, 0].astype(int)
    cx, cy, w, h = arr[:, 1], arr[:, 2], arr[:, 3], arr[:, 4]
    xyxy = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1)
    return cls, xyxy


# --------------------------------------------------------------------------- #
# Detection metrics (F1 at IoU 0.5, the RDD challenge metric)
# --------------------------------------------------------------------------- #
def box_iou_np(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Pairwise IoU between boxes a[n, 4] and b[m, 4] in xyxy format."""
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / np.maximum(union, 1e-12)


class DetectionEvaluator:
    """Accumulates predictions over a dataset and computes P / R / F1 at any
    confidence threshold.

    Predictions are matched to ground truth once (greedy, by descending confidence,
    same class, IoU >= iou_thr). Because greedy matching only depends on higher
    scoring boxes, the result for any threshold is just a filter on that single
    matching, so sweeping thresholds is cheap.

    IoU is computed on normalised coordinates, which gives the same value as pixel
    coordinates (IoU is invariant to scaling each axis).
    """

    def __init__(self, nc: int, iou_thr: float = 0.5):
        self.nc = nc
        self.iou_thr = iou_thr
        self.n_gt = np.zeros(nc, dtype=int)
        self._scores, self._cls, self._tp = [], [], []
        self._cache = None

    def add(self, pred_boxes, pred_scores, pred_cls, gt_boxes, gt_cls) -> None:
        self._cache = None
        pred_boxes = np.asarray(pred_boxes, dtype=float).reshape(-1, 4)
        pred_scores = np.asarray(pred_scores, dtype=float).reshape(-1)
        pred_cls = np.asarray(pred_cls).astype(int).reshape(-1)
        gt_boxes = np.asarray(gt_boxes, dtype=float).reshape(-1, 4)
        gt_cls = np.asarray(gt_cls).astype(int).reshape(-1)

        for c in range(self.nc):
            g = gt_boxes[gt_cls == c]
            self.n_gt[c] += len(g)
            m = pred_cls == c
            if not m.any():
                continue
            b, s = pred_boxes[m], pred_scores[m]
            order = np.argsort(-s)
            b, s = b[order], s[order]
            tp = np.zeros(len(b), dtype=bool)
            if len(g):
                ious = box_iou_np(b, g)
                matched = np.zeros(len(g), dtype=bool)
                for i in range(len(b)):
                    cand = np.where(matched, -1.0, ious[i])
                    j = int(cand.argmax())
                    if cand[j] >= self.iou_thr:
                        matched[j] = True
                        tp[i] = True
                        if matched.all():
                            break
            self._scores.append(s)
            self._cls.append(np.full(len(b), c, dtype=int))
            self._tp.append(tp)

    def _arrays(self):
        if self._cache is None:
            if self._scores:
                self._cache = (
                    np.concatenate(self._scores),
                    np.concatenate(self._cls),
                    np.concatenate(self._tp),
                )
            else:
                self._cache = (np.zeros(0), np.zeros(0, dtype=int), np.zeros(0, dtype=bool))
        return self._cache

    @staticmethod
    def _prf(tp: int, fp: int, fn: int) -> tuple[float, float, float]:
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        return p, r, f1

    def at_threshold(self, conf: float) -> dict:
        scores, cls, tp = self._arrays()
        keep = scores >= conf
        tp_c = np.bincount(cls[keep & tp], minlength=self.nc)
        fp_c = np.bincount(cls[keep & ~tp], minlength=self.nc)
        fn_c = self.n_gt - tp_c
        per_class = []
        for c in range(self.nc):
            p, r, f1 = self._prf(int(tp_c[c]), int(fp_c[c]), int(fn_c[c]))
            per_class.append(
                {"class_id": c, "precision": p, "recall": r, "f1": f1,
                 "tp": int(tp_c[c]), "fp": int(fp_c[c]), "fn": int(fn_c[c]),
                 "n_gt": int(self.n_gt[c])}
            )
        p, r, f1 = self._prf(int(tp_c.sum()), int(fp_c.sum()), int(fn_c.sum()))
        return {"conf": float(conf), "precision": p, "recall": r, "f1": f1,
                "tp": int(tp_c.sum()), "fp": int(fp_c.sum()), "fn": int(fn_c.sum()),
                "per_class": per_class}

    def average_precision(self) -> np.ndarray:
        """Per-class AP at the evaluator's IoU threshold (101-point interpolation, as in COCO)."""
        scores, cls, tp = self._arrays()
        ap = np.zeros(self.nc)
        recall_points = np.linspace(0, 1, 101)
        for c in range(self.nc):
            if self.n_gt[c] == 0:
                continue
            m = cls == c
            order = np.argsort(-scores[m], kind="stable")
            hits = tp[m][order]
            tpc = np.cumsum(hits)
            fpc = np.cumsum(~hits)
            recall = tpc / self.n_gt[c]
            precision = tpc / np.maximum(tpc + fpc, 1)
            # precision envelope, then sample it at fixed recall points
            envelope = np.maximum.accumulate(precision[::-1])[::-1] if len(precision) else precision
            idx = np.searchsorted(recall, recall_points, side="left")
            ap[c] = np.mean([envelope[i] if i < len(envelope) else 0.0 for i in idx])
        return ap

    def sweep(self, thresholds=DEFAULT_THRESHOLDS) -> list[dict]:
        return [self.at_threshold(t) for t in thresholds]

    def best(self, thresholds=DEFAULT_THRESHOLDS):
        """Returns (best_overall, best_per_class, all_rows)."""
        rows = self.sweep(thresholds)
        best = max(rows, key=lambda r: r["f1"])
        best_pc = []
        for c in range(self.nc):
            br = max(rows, key=lambda r: r["per_class"][c]["f1"])
            best_pc.append({"conf": br["conf"], **br["per_class"][c]})
        return best, best_pc, rows


# --------------------------------------------------------------------------- #
# Results logging
# --------------------------------------------------------------------------- #
def append_metrics_row(row: dict, path=None) -> Path:
    """Append one row to results/metrics.csv (created if missing)."""
    import pandas as pd

    path = Path(path) if path else PROJECT_ROOT / "results" / "metrics.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    new = pd.DataFrame([row])
    if path.exists() and path.stat().st_size > 0:
        new = pd.concat([pd.read_csv(path), new], ignore_index=True)
    new.to_csv(path, index=False)
    return path

# `fix_prepared_data.py`

```text
Fix an already-prepared data/{train,val,test}/{images,labels} copy of RDD2022.

Use this instead of convert_voc_to_yolo.py + make_split.py when the dataset was
downloaded pre-converted (e.g. from Kaggle). Two problems are fixed:

1. Class ids. The prepared labels use 5 ids: 0 D00, 1 D10, 2 D20, 3 "other
   corruption" (D43/D44/D50/Repair merged) and 4 D40 pothole. The per-class box
   totals (26016 / 11830 / 10617 / 6544 for ids 0/1/2/4) match the published RDD2022
   counts for D00 / D10 / D20 / D40. The project uses the 4 standard classes, so id 3
   boxes are dropped and id 4 becomes 3. This is only done when id 4 is present.

2. The split. The prepared split is random per image, so near-identical neighbouring
   frames land in train and val/test. Images are regrouped into blocks of consecutive
   file numbers per country (same heuristic as make_split.py) and whole blocks are
   assigned to one split.

In place (default): files are moved between the split folders, so no extra disk.
All original labels and the original split are first saved to
<root>/original_labels.zip. Running twice is refused unless --force is given.

    python scripts/fix_prepared_data.py --root data

To a new folder (--dst): the source is left untouched (e.g. a read-only Kaggle input).
Labels are written to <dst> and images are symlinked (copied if symlinks fail). If the
source was already fixed by this script, its split and classes are mirrored as they are.

    python scripts/fix_prepared_data.py --root /kaggle/input/rdd2022 --dst /kaggle/working/data
```

In [ ]:
import argparse
import os
import random
import re
import shutil
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

from tqdm import tqdm

from utils import IMG_EXTS, PROJECT_ROOT

In [ ]:
SPLITS = ("train", "val", "test")
CLASS_REMAP = {0: 0, 1: 1, 2: 2, 4: 3}  # source id -> project id; anything else is dropped
MARKER = ".fixed_by_fix_prepared_data"

In [ ]:
def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--root", default="data", help="folder holding train/, val/, test/")
    p.add_argument("--dst", default=None, help="write the fixed dataset here instead of changing --root")
    p.add_argument("--val-frac", type=float, default=0.15)
    p.add_argument("--test-frac", type=float, default=0.15)
    p.add_argument("--block-size", type=int, default=50, help="consecutive images treated as one sequence")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--keep-classes", action="store_true", help="only re-split, do not remap class ids")
    p.add_argument("--force", action="store_true", help="run even if the marker file says it already ran")
    return p.parse_args()

In [ ]:
def country_and_index(stem: str):
    m = re.match(r"(.+?)_(\d+)$", stem)
    return (m.group(1), int(m.group(2))) if m else (stem, None)

In [ ]:
def remap_label_text(text: str, dropped: Counter) -> str:
    out = []
    for line in text.splitlines():
        fields = line.split()
        if not fields:
            continue
        cid = int(float(fields[0]))
        if cid not in CLASS_REMAP:
            dropped[cid] += 1
            continue
        out.append(" ".join([str(CLASS_REMAP[cid])] + fields[1:]))
    return "\n".join(out) + ("\n" if out else "")

In [ ]:
def has_class_id(label_dirs, class_id: int) -> bool:
    prefix = f"{class_id} "
    for d in label_dirs:
        for f in d.glob("*.txt"):
            with open(f, encoding="utf-8") as fh:
                if any(line.startswith(prefix) for line in fh):
                    return True
    return False

In [ ]:
def link_or_copy(src: Path, dst: Path, state: dict) -> None:
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    if not state["copy"]:
        try:
            os.symlink(src.resolve(), dst)
            return
        except OSError:
            state["copy"] = True
            print("Symlinks are not available here; copying images instead.")
    shutil.copy2(src, dst)

In [ ]:
def block_split(items, args) -> dict:
    """Map each (country, block) group to a split, stratified by country."""
    groups = defaultdict(list)
    for item in items:
        country, idx = country_and_index(item[0])
        groups[(country, idx // args.block_size if idx is not None else item[0])].append(item)
    rng = random.Random(args.seed)
    by_country = defaultdict(list)
    for key in groups:
        by_country[key[0]].append(key)
    target = {}
    for country, keys in sorted(by_country.items()):
        keys = sorted(keys, key=str)
        rng.shuffle(keys)
        n = len(keys)
        n_test = max(1, round(n * args.test_frac)) if n >= 3 else 0
        n_val = max(1, round(n * args.val_frac)) if n >= 3 else 0
        for i, key in enumerate(keys):
            target[key] = "test" if i < n_test else ("val" if i < n_test + n_val else "train")
    return groups, target

In [ ]:
def main():
    args = parse_args()
    root = Path(args.root)
    if not root.is_absolute():
        root = PROJECT_ROOT / root
    dst = Path(args.dst) if args.dst else root
    if not dst.is_absolute():
        dst = PROJECT_ROOT / dst
    in_place = dst.resolve() == root.resolve()
    if (dst / MARKER).exists() and not args.force:
        sys.exit(f"{dst / MARKER} exists: this data was already fixed. Use --force to run again.")

    # ---- inventory ----
    items = []  # (stem, current_split, image_path)
    for s in SPLITS:
        img_dir = root / s / "images"
        if img_dir.is_dir():
            items += [(p.stem, s, p) for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    if not items:
        sys.exit(f"No images found under {root}/<split>/images")
    stems = [i[0] for i in items]
    if len(set(stems)) != len(stems):
        sys.exit("The same image name appears in more than one split; cannot re-split safely.")

    # splits.csv also counts: uploads to Kaggle can drop dotfiles such as the marker
    source_fixed = ((root / MARKER).exists() or (root / "splits.csv").exists()) and not in_place
    if source_fixed:
        print(f"{root} was already fixed by this script: mirroring its split and classes unchanged.")
        remap = False
    elif args.keep_classes:
        remap = False
    else:
        remap = has_class_id([root / s / "labels" for s in SPLITS if (root / s / "labels").is_dir()], 4)
        print("Class id 4 (D40 pothole) found: remapping to the 4 project classes." if remap else
              "No class id 4 found: labels already use the 4 project classes, keeping them.")

    # ---- backup (in place only; --dst never touches the source) ----
    if in_place:
        backup = root / "original_labels.zip"
        if not backup.exists():
            with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as zf:
                zf.writestr("original_split.csv", "stem,split\n" + "".join(f"{st},{sp}\n" for st, sp, _ in items))
                for stem, s, _ in tqdm(items, desc="Backing up labels"):
                    lbl = root / s / "labels" / f"{stem}.txt"
                    if lbl.exists():
                        zf.write(lbl, f"{s}/labels/{stem}.txt")
            print(f"Backed up original labels and split to {backup}")

    # ---- split ----
    if source_fixed:
        groups = {(stem, None): [(stem, s, img)] for stem, s, img in items}
        target = {key: members[0][1] for key, members in groups.items()}
    else:
        groups, target = block_split(items, args)

    for s in SPLITS:
        (dst / s / "images").mkdir(parents=True, exist_ok=True)
        (dst / s / "labels").mkdir(parents=True, exist_ok=True)

    dropped, moved, state = Counter(), 0, {"copy": False}
    class_counts = {s: Counter() for s in SPLITS}
    rows = []
    for key, members in tqdm(sorted(groups.items(), key=lambda kv: str(kv[0])), desc="Writing split"):
        dst_split = target[key]
        for stem, src_split, img in members:
            src_lbl = root / src_split / "labels" / f"{stem}.txt"
            text = src_lbl.read_text(encoding="utf-8") if src_lbl.exists() else ""
            if remap:
                text = remap_label_text(text, dropped)
            (dst / dst_split / "labels" / f"{stem}.txt").write_text(text, encoding="utf-8")
            if in_place:
                if src_split != dst_split:
                    if src_lbl.exists():
                        src_lbl.unlink()
                    os.replace(img, dst / dst_split / "images" / img.name)
                    moved += 1
            else:
                link_or_copy(img, dst / dst_split / "images" / img.name, state)
                moved += src_split != dst_split
            for line in text.splitlines():
                class_counts[dst_split][int(float(line.split()[0]))] += 1
            rows.append(f"{stem},{key[0]},{key[0]}_{key[1]},{dst_split}\n")

    (dst / "splits.csv").write_text("stem,country,group,split\n" + "".join(sorted(rows)), encoding="utf-8")
    (dst / MARKER).write_text(f"block_size={args.block_size} seed={args.seed} remap={remap}\n", encoding="utf-8")

    # ---- summary ----
    print(f"\n{moved} images changed split. Dropped boxes by source class: {dict(dropped)}")
    per_split = Counter(target[k] for k, m in groups.items() for _ in m)
    for s in SPLITS:
        print(f"{s:>5}: {per_split[s]:>6} images | boxes per class {dict(sorted(class_counts[s].items()))}")
    print(f"\nWrote {dst / 'splits.csv'}")

## Arguments

Edit `ARGS` (the command-line arguments of the script), then run the next cell.

In [ ]:
import sys

ARGS = []
sys.argv = ['fix_prepared_data.py'] + ARGS

In [ ]:
main()